# StudentInsight: Google Colab and Jupyter

Explore the same public datasets, predictors, calibration and evaluation used by StudentInsight.
Run the cells in order. CPU is sufficient; no GPU is needed.

- **Google Colab:** open this notebook, connect to a runtime, then choose **Runtime > Run all**. Setup clones the GitHub project and installs its dependencies into that runtime.
- **Local Jupyter:** launch `run-notebook.ps1` from the project and choose the **StudentInsight (.venv)** kernel. Alternatively install `requirements-notebooks.txt` into your kernel's Python environment.
- This notebook works directly with the project's Python modules. The Streamlit login/dashboard remains a separate interface. It does not connect to your local account database or synchronize the deployed app.

The default analysis uses only the bundled UCI records. Outputs are historical evaluations, not live student records or proof of reliability at your institution. Colab runtime files are temporary: download results before disconnecting. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html) and [Jupyter installation guide](https://jupyter.org/install).


## 1. Locate the project and prepare the notebook environment


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPOSITORY = "https://github.com/GamerStudioszw/Student-Performance-Prediction.git"
BRANCH = "main"
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/Student-Performance-Prediction")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPOSITORY, str(PROJECT_ROOT)], check=True)
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = next((path for path in (current, *current.parents)
                         if (path / "model.py").is_file()
                         and (path / "requirements.txt").is_file()), None)
    if PROJECT_ROOT is None:
        raise RuntimeError("Open this notebook inside the StudentInsight repository.")

if not (PROJECT_ROOT / "model_service.py").is_file():
    raise RuntimeError("The selected folder is not the StudentInsight project.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Install into the current Colab kernel before importing numerical libraries.
# Local dependencies are installed by the launcher, avoiding repeated pip changes.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)

print("Environment:", "Google Colab" if IN_COLAB else "Local Jupyter")
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)
if (PROJECT_ROOT / ".git").exists():
    print("Git revision:", subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


If package installation requests a runtime restart, restart the kernel/runtime and run all cells again. In Colab, an existing checkout is reused without overwriting your edits; start a fresh runtime to clone the latest GitHub version.


In [ ]:
import json
import altair as alt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, FileLink

from model import (SCHEMA_VERSION, PUBLIC_SUBJECTS, FEATURE_SETS, load_data,
                   train_and_evaluate, prediction_report, alert_evaluation)
from model_service import SOURCE, file_hash, finalize_bundle, save_bundle
from cohort import export_csv

if SCHEMA_VERSION < 3:
    raise RuntimeError("This checkout predates the predictor repairs. Use the current main branch.")
print("scikit-learn:", sklearn.__version__)
print("Source:", SOURCE)
datasets = {subject: load_data(PROJECT_ROOT / "data" / f"student-{subject}.csv")
            for subject in PUBLIC_SUBJECTS}
display(pd.DataFrame([{"Subject": PUBLIC_SUBJECTS[key], "Records": len(frame),
                       "Grade scale": "0-20", "Prediction cutoff": "After period two"}
                      for key, frame in datasets.items()]))


## 2. Train and compare both bundled subjects

This explicitly runs a new experiment **in memory**, using the same six candidates, training folds, fixed seed, simplicity rule and separate calibration/test partitions as the application. It does not overwrite historical artifacts or change the deployed predictor.

The existing test results already informed the repairs: these comparisons are **not independent confirmation**. The G2 persistence baseline is allowed to win. All required historical values must be recorded; missing inputs are never invented or imputed.


In [ ]:
bundles = {}
for subject, frame in datasets.items():
    path = PROJECT_ROOT / "data" / f"student-{subject}.csv"
    bundles[subject] = finalize_bundle(
        *train_and_evaluate(frame), subject=subject,
        fingerprint=file_hash(path), source=SOURCE)

summary = pd.DataFrame([
    {"Subject": PUBLIC_SUBJECTS[subject], "Predictor": metrics["selected_model"],
     "Feature set": metrics["feature_set"], "Required inputs": ", ".join(metrics["features"]),
     "Test MAE": metrics["mae"], "Test RMSE": metrics["rmse"], "Test R2": metrics["r2"],
     "G2 baseline MAE": metrics["g2_baseline_mae"],
     "Nominal coverage": metrics["interval"]["nominal_coverage"],
     "Observed test coverage": metrics["interval"]["test_coverage"],
     "Average interval width": metrics["interval"]["average_width"],
     "Calibration records": metrics["calibration_rows"], "Test records": metrics["test_rows"]}
    for subject, (_, metrics, _) in bundles.items()
])
display(summary)


## 3. Inspect a subject, candidate comparisons and charts

Set `SUBJECT` to `"mat"` or `"por"`, then rerun this cell and the cells below it. The comparison table uses cross-validation MAE on fitting records. The chart uses the separate historical test partition. MAE/RMSE are grade-point errors; R2 is not an accuracy percentage.


In [ ]:
SUBJECT = "mat"  # "mat": Mathematics; "por": Portuguese language
if SUBJECT not in bundles:
    raise ValueError("Choose mat or por; other subjects require suitable completed history.")
fitted, metrics, test_results = bundles[SUBJECT]
print(PUBLIC_SUBJECTS[SUBJECT], "|", metrics["predictor_kind"])
print("Selection rule:", metrics["selection_rule"])
comparison = pd.DataFrame(metrics["comparison"])
display(comparison[["model", "feature_set", "features", "cv_mae", "cv_std"]])
display(alt.Chart(comparison).mark_bar().encode(
    x=alt.X("cv_mae:Q", title="Cross-validation MAE (grade points)"),
    y=alt.Y("model:N", sort="x", title="Candidate"),
    tooltip=["model", "feature_set", "cv_mae", "cv_std"]
).properties(title=f"{PUBLIC_SUBJECTS[SUBJECT]}: validation comparison", height=230))

display(alt.Chart(test_results).mark_circle(size=65, opacity=0.5).encode(
    x=alt.X("actual_grade:Q", title="Recorded final grade G3"),
    y=alt.Y("predicted_grade:Q", title="Predicted final grade G3"),
    tooltip=["source_row", "actual_grade", "predicted_grade", "interval_lower", "interval_upper"]
).properties(title="Historical test records: recorded versus predicted", height=300))


## 4. Prediction intervals and below-reference alerts

Intervals use the finite-sample split-conformal absolute-error quantile. A single calibration radius gives **constant width**, not personalized uncertainty. Raw predictions and intervals are not clipped. An 80% nominal coverage target is not an individual's probability of passing. Realized coverage can differ from the target.

For alerts, the positive outcome is **recorded G3 < reference grade** and the alert is **predicted G3 < the same reference**. Equality is not below reference. Undefined scores are displayed as missing, not zero. These are historical test metrics, separate from uploaded class performance.


In [ ]:
REFERENCE_GRADE = 10
interval = metrics["interval"]
display(pd.DataFrame([interval]))
alerts = alert_evaluation(test_results.actual_grade, test_results.predicted_grade, REFERENCE_GRADE)
display(pd.DataFrame([alerts]))
display(pd.DataFrame(
    [[alerts["tp"], alerts["fn"]], [alerts["fp"], alerts["tn"]]],
    index=["Recorded G3 below reference", "Recorded G3 at/above reference"],
    columns=["Alert", "No alert"]))


## 5. Predict from recorded inputs

Use a real bundled held-out row as an example. Only the selected predictor's required columns are passed; optional G1 is retained for observed progress. G3 is never passed to prediction. To try your own profile, replace `profile` with a one-row DataFrame containing the displayed required inputs, using the same whole-number 0-20 grade scale.

G2 - G1 describes recorded progress, not independent information or a causal effect. Percentage equivalent means grade multiplied by five, not probability of passing.


In [ ]:
print("Required prediction inputs:", metrics["features"])
source_row = int(test_results.iloc[0]["source_row"])
profile_columns = list(dict.fromkeys([*metrics["features"], "G1"]))
profile = datasets[SUBJECT].loc[[source_row], profile_columns].copy()
report = prediction_report(fitted, profile, metrics, threshold=REFERENCE_GRADE)
display(report)


## 6. Export this experiment

Create a new folder under `notebook-results/` (excluded from Git). This saves both subjects' new evaluation JSON/CSV files, the summary, selected-subject profile report, alert evaluation and environment provenance. It does not replace saved application bundles. `FileLink` works in local Jupyter; set `DOWNLOAD_IN_COLAB = True` to download a ZIP in Colab. No Google Drive mount or credentials are required.


In [ ]:
from datetime import datetime, timezone
import importlib.metadata
import shutil
import uuid

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
export_dir = PROJECT_ROOT / "notebook-results" / run_id
export_dir.mkdir(parents=True, exist_ok=False)
summary.to_csv(export_dir / "summary.csv", index=False)
for subject, (_, subject_metrics, results) in bundles.items():
    (export_dir / f"{subject}_metrics.json").write_text(
        json.dumps(subject_metrics, indent=2, allow_nan=False), encoding="utf-8")
    results.to_csv(export_dir / f"{subject}_test_predictions.csv", index=False)
(export_dir / f"{SUBJECT}_profile.csv").write_bytes(export_csv(report))
pd.DataFrame([{"subject": SUBJECT, "bundle_id": metrics["bundle_id"],
               "evaluation_population": "Historical test partition", **alerts}]).to_csv(
    export_dir / f"{SUBJECT}_alerts.csv", index=False)
provenance = {"python": sys.version, "environment": "Colab" if IN_COLAB else "Local Jupyter",
              "packages": {name: importlib.metadata.version(name)
                           for name in ["numpy", "pandas", "scikit-learn", "joblib", "altair"]}}
(export_dir / "environment.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(export_dir), "zip", root_dir=export_dir))
print("Saved experiment:", export_dir)
DOWNLOAD_IN_COLAB = False
if IN_COLAB:
    print("ZIP:", archive, "- set DOWNLOAD_IN_COLAB = True to download.")
    if DOWNLOAD_IN_COLAB:
        from google.colab import files
        files.download(str(archive))
else:
    display(FileLink(archive.relative_to(PROJECT_ROOT).as_posix(), url_prefix="/files/"))


## 7. Optional: save a local application bundle

Leave `SAVE_APPLICATION_BUNDLE = False` for exploration. Setting it to `True` saves the selected subject as a new immutable run and updates its local `latest.json` pointer using the same artifact service as `train.py`. Repeat the training cell before saving the same bundle a second time. In Colab, this affects only the temporary runtime checkout; it does not deploy a model to your PC or the hosted app. Model binaries are excluded from Git.

For additional subjects, use `subject_training.train_subject` with genuine completed history and the chosen versioned input schema. Do not relabel Mathematics/Portuguese models or fill missing historical parameters. The Halabja download is not integrated.


In [ ]:
SAVE_APPLICATION_BUNDLE = False
if SAVE_APPLICATION_BUNDLE:
    saved_directory = save_bundle(bundles[SUBJECT])
    print("Saved new application bundle:", saved_directory)
else:
    print("Application artifact pointers unchanged.")
